In [ ]:
################## 63 datapoint of mixed an and non ####################

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
#import mordred                                   
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler, RobustScaler, MaxAbsScaler, QuantileTransformer, PowerTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Define the folder and file path
folder_name = "result"
file_name_RDkit_mix = "Mixed_exclude.xlsx"  # Replace with your actual file name


file_path_RDkit_mix = os.path.join(folder_name, file_name_RDkit_mix)

# Load the Excel file into a DataFrame
data_RDkit_mix = pd.read_excel(file_path_RDkit_mix)


miles_column_RDkit_mix = data_RDkit_mix['SMILES']
EO_PO = data_RDkit_mix[['EO', 'PO']]
print(EO_PO.head())

# Display the first few rows of the DataFrame
print(data_RDkit_mix.head())

In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors.MoleculeDescriptors import MolecularDescriptorCalculator

def RDKit_descriptors(smiles_list):
    # build a calculator over the standard 1D/2D descriptors
    names = [name for name, fn in Descriptors.descList]
    calc  = MolecularDescriptorCalculator(names)

    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            # failed parse → fill with Nones
            rows.append([None]*len(names))
            continue

        # optional: sanitize explicitly
        try:
            Chem.SanitizeMol(mol)
        except:
            rows.append([None]*len(names))
            continue

        # *** DO NOT ADD explicit H’s here ***
        # mol = Chem.AddHs(mol)

        # compute the ~200 descriptors
        vals = list(calc.CalcDescriptors(mol))
        rows.append(vals)

    # return a DataFrame
    return pd.DataFrame(rows, columns=names)

# usage
df = pd.read_excel("result/Mixed_exclude.xlsx")
#df = df.drop(['Head','Tail'], axis=1)
desc_df = RDKit_descriptors(df["SMILES"])
Mol_descriptors = pd.concat([df, desc_df], axis=1)
Mol_descriptors.to_excel("Case4_1.xlsx", index=False)
print("Done.")


In [ ]:
# Load the saved Excel to verify
df_RDK_mix = pd.read_excel('Case4_1.xlsx')
print(df_RDK_mix.head())
print(df_RDK_mix.info())


In [ ]:
# Define feature matrix X and target vector y
X_RDkit_mix = df_RDK_mix.drop(['Cc','SMILES','EO','PO'], axis=1)  # Features include SMILES and descriptors
y_RDkit_mix = df_RDK_mix['Cc']  # Target variable

# Display the first few rows
print(X_RDkit_mix.head())
print(y_RDkit_mix.head())


In [ ]:


# Identify numeric columns
numeric_cols_RDkit_mix = X_RDkit_mix.select_dtypes(include=['number']).columns.tolist()

# Identify non-numeric columns
non_numeric_cols_RDkit_mix = X_RDkit_mix.select_dtypes(exclude=['number']).columns.tolist()

print(f"Numeric Columns: {numeric_cols_RDkit_mix}")
print(f"Non-Numeric Columns: {non_numeric_cols_RDkit_mix}")


In [ ]:
# Check the data types of the numeric columns
print("Data Types of Numeric Columns:")
print(X_RDkit_mix.dtypes)


In [ ]:
# Identify non-numeric columns in numeric_cols
non_numeric_in_numeric_RDkit_mix = X_RDkit_mix.select_dtypes(exclude=['number']).columns.tolist()

print(f"Non-Numeric Columns in Numeric Features: {non_numeric_in_numeric_RDkit_mix}")


In [ ]:
# Attempt to convert non-numeric columns to numeric, coercing errors to NaN
for col in non_numeric_in_numeric_RDkit_mix:
    X_RDkit_mix[col] = pd.to_numeric(X_RDkit_mix[col], errors='coerce')
    print(f"Converted column '{col}' to numeric.")


In [ ]:
# Re-check the data types after conversion
print("Data Types After Conversion:")
print(X_RDkit_mix.dtypes)


In [ ]:
# Identify columns with all NaN values
cols_all_nan_RDkit_mix = [col for col in non_numeric_in_numeric_RDkit_mix if X_RDkit_mix[col].isna().all()]
print(f"\nColumns with all NaNs: {cols_all_nan_RDkit_mix}")

# Drop these columns from X_numeric
if cols_all_nan_RDkit_mix:
    X_numeric_RDkit_mix = X_RDkit_mix.drop(columns=cols_all_nan_RDkit_mix)
    print(f"Dropped columns with all NaNs: {cols_all_nan_RDkit_mix}")



In [ ]:
# Re-check data types after conversion
print("\nData Types After Conversion:")
print(X_RDkit_mix.dtypes)


In [ ]:
# First Code Snippet
cols_all_nan_RDkit_mix = [col for col in non_numeric_in_numeric_RDkit_mix if X_RDkit_mix[col].isna().all()]
print(f"\nColumns with all NaNs: {cols_all_nan_RDkit_mix}")

# Here you define X_numeric_RDkit_mix
if cols_all_nan_RDkit_mix:
    X_numeric_RDkit_mix = X_RDkit_mix.drop(columns=cols_all_nan_RDkit_mix)
    print(f"Dropped columns with all NaNs: {cols_all_nan_RDkit_mix}")
else:
    # If there are no all-NaN columns, you should still define X_numeric_RDkit_mix
    X_numeric_RDkit_mix = X_RDkit_mix.copy()


In [ ]:
# Update numeric_cols by removing the dropped columns
numeric_cols_updated_RDkit_mix = [col for col in numeric_cols_RDkit_mix if col not in cols_all_nan_RDkit_mix]
print(f"Updated Numeric Columns after dropping all-NaN columns: {numeric_cols_updated_RDkit_mix}")


In [ ]:
# confirm again
cols_all_nan_RDkit_mix = [col for col in X_numeric_RDkit_mix if X_numeric_RDkit_mix[col].isna().all()]
print(f"Columns with all NaNs: {cols_all_nan_RDkit_mix}")

In [ ]:
X_minmax_scaled_df_RDkit_mix_1 = X_numeric_RDkit_mix.dropna(axis=1, how='all')


In [ ]:
# Visualize the reduced correlation matrix
Based = X_minmax_scaled_df_RDkit_mix_1.corr()
plt.rcParams["font.family"] = "Times New Roman"
# Plot the correlation heatmap with increased annotation font size
plt.figure(figsize=(120, 100))  # Set figure size for better readability
ax = sns.heatmap(Based, 
            annot=True, 
            annot_kws={"size": 20},  # Increase annotation font size
            cmap='coolwarm', 
            fmt=".2f", 
            linewidths=0.5, 
            cbar=True, 
            square=True)


# Increase the font size of the x and y axis tick labels
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=100)
# Optionally, increase font size of axis labels and title
plt.xlabel('Descriptors', fontsize=110)
plt.ylabel('Descriptors', fontsize=110)
plt.title('Correlation Heatmap of RDKit Descriptors', fontsize=130)

In [ ]:

from sklearn.feature_selection import VarianceThreshold

# Check for constant features
constant_features = [col for col in X_minmax_scaled_df_RDkit_mix_1.columns if X_minmax_scaled_df_RDkit_mix_1[col].nunique() == 1]
print(f"Constant features to remove: {constant_features}")

# Drop constant features
X_reduced_RDkit_mix_complete = X_minmax_scaled_df_RDkit_mix_1.drop(columns=constant_features)

# Set variance threshold
threshold = 0.0  # Adjust as needed

# Apply VarianceThreshold
selector = VarianceThreshold(threshold)
X_low_variance_complete = selector.fit_transform(X_reduced_RDkit_mix_complete)

# Get remaining features
remaining_features = X_reduced_RDkit_mix_complete.columns[selector.get_support()]
print(f"Remaining features after removing low variance: {remaining_features.tolist()}")


# Visualize the reduced correlation matrix
reduced_corr_matrix = X_reduced_RDkit_mix_complete.corr()

# Plot the correlation heatmap with increased annotation font size
plt.figure(figsize=(120, 100))  # Set figure size for better readability
ax = sns.heatmap(reduced_corr_matrix, 
            annot=True, 
            annot_kws={"size": 20},  # Increase annotation font size
            cmap='coolwarm', 
            fmt=".2f", 
            linewidths=0.5, 
            cbar=True, 
            square=True)

# Increase the font size of the x and y axis tick labels
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=100)
# Optionally, increase font size of axis labels and title
plt.xlabel('Descriptors', fontsize=110)
plt.ylabel('Descriptors', fontsize=110)
plt.title('Correlation Heatmap of RDKit Descriptors', fontsize=130)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Compute the correlation matrix
corr_matrix_RDkit_mix_1 = X_reduced_RDkit_mix_complete.corr().abs()

# Select upper triangle of correlation matrix
upper_RDkit_mix_1 = corr_matrix_RDkit_mix_1.where(np.triu(np.ones(corr_matrix_RDkit_mix_1.shape), k=1).astype(bool))

# Find features with correlation greater than 0.8
to_drop_RDkit_mix_1 = [column for column in upper_RDkit_mix_1.columns if any(upper_RDkit_mix_1[column] > abs(0.8))]

print(f"\nFeatures to drop due to high correlation: {to_drop_RDkit_mix_1}")

# Drop the features
X_reduced_RDkit_mix = X_reduced_RDkit_mix_complete.drop(columns=to_drop_RDkit_mix_1)

print(f"\nFeature matrix shape after dropping correlated features: {X_reduced_RDkit_mix.shape}")


# Visualize the reduced correlation matrix
reduced_corr_matrix = X_reduced_RDkit_mix.corr()

# Plot the correlation heatmap with increased annotation font size
plt.figure(figsize=(120, 100))  # Set figure size for better readability
ax = sns.heatmap(reduced_corr_matrix, 
            annot=True, 
            annot_kws={"size": 40},  # Increase annotation font size
            cmap='coolwarm', 
            fmt=".2f", 
            linewidths=0.5, 
            cbar=True, 
            square=True)

# Increase the font size of the x and y axis tick labels
plt.xticks(fontsize=20)
plt.yticks(fontsize=20, rotation = 0)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=100)
# Optionally, increase font size of axis labels and title
plt.xlabel('Descriptors', fontsize=110)
plt.ylabel('Descriptors', fontsize=110)
plt.title('Correlation Heatmap of RDKit Descriptors', fontsize=130)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate the correlation matrix
correlation_matrix_reduce = X_reduced_RDkit_mix.corr()

# Plot the correlation heatmap with annotations
plt.figure(figsize=(120, 100))  # Set figure size for better readability
sns.heatmap(correlation_matrix_reduce, annot=True, cmap='coolwarm', fmt=".2f", 
            linewidths=0.5, cbar=True, square=True)
plt.title('Correlation Heatmap of Features')
plt.show()


In [ ]:
from sklearn.feature_selection import VarianceThreshold
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Compute the correlation matrix
corr_matrix_RDkit_mix_complete = X_minmax_scaled_df_RDkit_mix_1.corr()

# Select upper triangle of correlation matrix
upper_RDkit_mix_complete = corr_matrix_RDkit_mix_complete.where(np.triu(np.ones(corr_matrix_RDkit_mix_complete.shape), k=1).astype(bool))

# Find features with correlation greater than 0.8 or less than -0.8
to_drop_high_corr = [column for column in upper_RDkit_mix_complete.columns if any(upper_RDkit_mix_complete[column] > 0.8)]
to_drop_low_corr = [column for column in upper_RDkit_mix_complete.columns if any(upper_RDkit_mix_complete[column] < -0.8)]

# Combine all features to drop
to_drop_RDkit_mix_complete = list(set(to_drop_high_corr + to_drop_low_corr))

print(f"\nFeatures to drop due to high or low correlation: {to_drop_RDkit_mix_complete}")

# Drop the features
X_reduced_RDkit_complete =X_minmax_scaled_df_RDkit_mix_1.drop(columns=to_drop_RDkit_mix_complete)

print(f"\nFeature matrix shape after dropping correlated features: {X_reduced_RDkit_complete.shape}")

# Check for constant features
constant_features = [col for col in X_reduced_RDkit_complete.columns if X_reduced_RDkit_complete[col].nunique() == 1]
print(f"Constant features to remove: {constant_features}")

# Drop constant features
X_reduced_RDkit_mix_complete = X_reduced_RDkit_complete.drop(columns=constant_features)

# Set variance threshold
threshold = 0.0  # Adjust as needed

# Apply VarianceThreshold
selector = VarianceThreshold(threshold)
X_low_variance_complete = selector.fit_transform(X_reduced_RDkit_mix_complete)

# Get remaining features
remaining_features = X_reduced_RDkit_mix_complete.columns[selector.get_support()]
print(f"Remaining features after removing low variance: {remaining_features.tolist()}")


# Visualize the reduced correlation matrix
reduced_corr_matrix = X_reduced_RDkit_mix_complete.corr()

# Plot the correlation heatmap with increased annotation font size
plt.figure(figsize=(120, 100))  # Set figure size for better readability
sns.heatmap(reduced_corr_matrix, 
            annot=True, 
            annot_kws={"size": 40},  # Increase annotation font size
            cmap='coolwarm', 
            fmt=".2f", 
            linewidths=0.5, 
            cbar=True, 
            square=True)

# Increase the font size of the x and y axis tick labels
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)

# Optionally, increase font size of axis labels and title
plt.xlabel('Features', fontsize=30)
plt.ylabel('Features', fontsize=30)
plt.title('Correlation Heatmap of Features', fontsize=40)



In [ ]:


print(len(to_drop_high_corr))
print(len(to_drop_low_corr))
print(len(to_drop_RDkit_mix_complete))

In [ ]:
# Compute the correlation matrix
corr_matrix_RDkit_mix_complete = X_reduced_RDkit_mix_complete.corr()

# Select lower triangle of correlation matrix
lower_RDkit_mix_complete = corr_matrix_RDkit_mix_complete.where(
    np.tril(np.ones(corr_matrix_RDkit_mix_complete.shape), k=-1).astype(bool)
)

# Visualize the lower triangle correlation matrix
plt.figure(figsize=(120, 100))  # Set figure size for better readability
sns.heatmap(lower_RDkit_mix_complete, 
            annot=True, 
            annot_kws={"size": 40},  # Increase annotation font size
            cmap='coolwarm', 
            fmt=".2f", 
            linewidths=0.5, 
            cbar=True,
            cbar_kws={'shrink': 1, 'label': 'Correlation Coefficient', 'format': '%.2f'},  # Increase cbar font size 
            square=True)

# Increase the font size of the x and y axis tick labels
# Rotate x and y tick labels
plt.xticks(rotation=45, fontsize=24)  # Rotate x-axis labels 45 degrees
plt.yticks(rotation=0, fontsize=24)   # Rotate y-axis labels horizontally

# Customize the color bar font size
cbar = plt.gcf().axes[-1]  # Get the color bar axis
cbar.tick_params(labelsize=40)  # Set the font size for color bar ticks

# Optionally, increase font size of axis labels and title
plt.xlabel('Features', fontsize=40)
plt.ylabel('Features', fontsize=40)
plt.title('Lower Half Correlation Heatmap of Features', fontsize=60)
plt.show()


In [ ]:
print(f"\nFeatures dropped: {to_drop_RDkit_mix_1}")
print(f"Remaining features: {X_reduced_RDkit_mix.shape[1]}")


print(f"Remaining features: {X_reduced_RDkit_mix_complete.shape[1]}")

In [ ]:
print(f"Number of samples in X_reduced: {X_reduced_RDkit_mix.shape[0]}")
print(f"Number of samples in X_reduced_RDkit_mix_completed: {X_reduced_RDkit_mix_complete.shape[0]}")
print(f"Number of samples in y: {y_RDkit_mix.shape[0]}")


In [ ]:
print(f"Indices in X_reduced: {X_reduced_RDkit_mix.index}")
print(f"Indices in X_reduced_RDkit_mix_complete: {X_reduced_RDkit_mix_complete.index}")

print(f"Indices in y: {y_RDkit_mix.index}")


In [ ]:
# Assuming X_final is your current feature matrix after imputation and optional dropping
combined_df_RDkit_mix = pd.concat([X_reduced_RDkit_mix, y_RDkit_mix], axis=1)

print(f"Combined DataFrame shape: {combined_df_RDkit_mix.shape}")
print(f"Columns: {combined_df_RDkit_mix.columns.tolist()}")

combined_df_RDkit_mix_complete = pd.concat([X_reduced_RDkit_mix_complete, y_RDkit_mix], axis=1)

print(f"Combined DataFrame shape: {combined_df_RDkit_mix_complete.shape}")
print(f"Columns: {combined_df_RDkit_mix_complete.columns.tolist()}")


In [ ]:
# Separate features and target
X_cleaned_RDkit_mix_1 = combined_df_RDkit_mix.drop(['Cc'], axis=1)
y_cleaned_RDkit_mix = combined_df_RDkit_mix['Cc']

print(f"Number of samples in X_cleaned: {X_cleaned_RDkit_mix_1.shape[0]}")
print(f"Number of samples in y_cleaned: {y_cleaned_RDkit_mix.shape[0]}")


X_cleaned_RDkit_mix_1_complete = combined_df_RDkit_mix_complete.drop(['Cc'], axis=1)
y_cleaned_RDkit_mix_complete = combined_df_RDkit_mix_complete['Cc']

print(f"Number of samples in X_cleaned_complete: {X_cleaned_RDkit_mix_1_complete.shape[0]}")
print(f"Number of samples in y_cleaned_complete: {y_cleaned_RDkit_mix_complete.shape[0]}")


In [ ]:
# Check for missing values in y_cleaned
print("Missing values in y_cleaned:")
print(y_cleaned_RDkit_mix.isnull().sum())

print("Missing values in y_cleaned_complete:")
print(y_cleaned_RDkit_mix_complete.isnull().sum())

In [ ]:
# Identify indices where y_cleaned is NaN
nan_indices_RDkit_mix = y_cleaned_RDkit_mix[y_cleaned_RDkit_mix.isnull()].index
print(f"Indices with NaN in y_cleaned: {nan_indices_RDkit_mix.tolist()}")

nan_indices_RDkit_mix_complete = y_cleaned_RDkit_mix_complete[y_cleaned_RDkit_mix_complete.isnull()].index
print(f"Indices with NaN in y_cleaned_complete: {nan_indices_RDkit_mix_complete.tolist()}")


In [ ]:
# Drop samples with NaN in y_cleaned
X_cleaned_RDkit_mix = X_cleaned_RDkit_mix_1.drop(index=nan_indices_RDkit_mix).reset_index(drop=True)
y_cleaned_RDkit_mix = y_cleaned_RDkit_mix.drop(index=nan_indices_RDkit_mix).reset_index(drop=True)

print(f"Shape of X_cleaned after dropping NaNs: {X_cleaned_RDkit_mix.shape}")
print(f"Shape of y_cleaned after dropping NaNs: {y_cleaned_RDkit_mix.shape}")


# Drop samples with NaN in y_cleaned_complete
X_cleaned_RDkit_mix_complete = X_cleaned_RDkit_mix_1_complete.drop(index=nan_indices_RDkit_mix_complete).reset_index(drop=True)
y_cleaned_RDkit_mix_complete = y_cleaned_RDkit_mix_complete.drop(index=nan_indices_RDkit_mix_complete).reset_index(drop=True)

print(f"Shape of X_cleaned_complete after dropping NaNs: {X_cleaned_RDkit_mix_complete.shape}")
print(f"Shape of y_cleaned_complete after dropping NaNs: {y_cleaned_RDkit_mix_complete.shape}")


In [ ]:


# Example: Combining original data with descriptors
# Assuming 'df' is your original DataFrame with 'SMILES' and 'Cc'
# 'df_descriptors' is the DataFrame with molecular descriptors

df_cleaned_RDkit_mix = pd.concat([y_cleaned_RDkit_mix, X_cleaned_RDkit_mix], axis=1)

# Display the first few rows to verify
print(df_cleaned_RDkit_mix.head())
# Define the output file path
output_excel_path_clean_RDkit_mix = 'descriptors_cleaned_RDkit_mix_Exclude.xlsx'

# Save the DataFrame to Excel
df_cleaned_RDkit_mix.to_excel(output_excel_path_clean_RDkit_mix, index=False)

print(f"Data successfully saved to {output_excel_path_clean_RDkit_mix}")



df_cleaned_RDkit_mix_complete = pd.concat([y_cleaned_RDkit_mix_complete, X_cleaned_RDkit_mix_complete], axis=1)

# Display the first few rows to verify
print(df_cleaned_RDkit_mix_complete.head())
# Define the output file path
output_excel_path_clean_RDkit_mix_complete = 'Case4_1_clean.xlsx'

# Save the DataFrame to Excel
df_cleaned_RDkit_mix_complete.to_excel(output_excel_path_clean_RDkit_mix_complete, index=False)

print(f"Data successfully saved to {output_excel_path_clean_RDkit_mix_complete}")

In [ ]:
# Load the saved Excel to verify
df_RDK_mix_ = pd.read_excel('Case4_1_clean.xlsx')
print(df_RDK_mix_.head())
print(df_RDK_mix_.info())

# Define feature matrix X and target vector y
X_cleaned_RDkit_mix_complete = df_RDK_mix_.drop(['Cc'], axis=1)  # Features include SMILES and descriptors
y_cleaned_RDkit_mix_complete = df_RDK_mix_['Cc']  # Target variable

# Display the first few rows
print(X_cleaned_RDkit_mix_complete.head())
print(y_cleaned_RDkit_mix_complete.head())

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, MaxAbsScaler, QuantileTransformer, PowerTransformer
from sklearn.pipeline import Pipeline

# Initialize the Min-Max Scaler
#scaler = MinMaxScaler()
scaler =StandardScaler()

pipeline = Pipeline([
    ('Standard', StandardScaler())
    
])

# Fit the scaler on the feature matrix and transform
X_minmax_scaled_RDkit_mix_1 = pipeline.fit_transform(X_cleaned_RDkit_mix_complete)

# Convert back to DataFrame for easier handling
X_cleaned_RDkit_mix_complete1 = pd.DataFrame(X_minmax_scaled_RDkit_mix_1, columns=X_cleaned_RDkit_mix_complete.columns)

print("\nMin-Max Normalized Features:")
print(X_cleaned_RDkit_mix_complete1.head())

# Distribution plot before and after transformation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

# 1) Set font globally
plt.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.linewidth": 1.0,
})

# 2) Seaborn style (won't change font)
sns.set_theme(style="white", context="paper")

y = y_cleaned_RDkit_mix_complete.dropna()

fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=300)

sns.histplot(
    y,
    bins="fd",
    kde=True,
    kde_kws={"bw_adjust": 1.1, "cut": 0},
    line_kws={"linewidth": 1.6},
    edgecolor="white",
    linewidth=0.8,
    alpha=0.9,
    ax=ax
)

ax.set_title("Distribution of Target Variable", pad=8)
ax.set_xlabel("Cc")
ax.set_ylabel("Count")

ax.grid(False)
ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
ax.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=6))

sns.despine(ax=ax)
fig.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams["font.family"] = "Times New Roman"
# Adjust the number of bins
sns.histplot(y_cleaned_RDkit_mix_complete, kde=True, bins=15, line_kws={'color':'red'})  # Adjust bins (e.g., 15 for larger steps)
plt.title('Distribution of Target Variable')
plt.xlabel('Cc')
plt.ylabel('Count')
# Customize the x-axis ticks
min_value = y_cleaned_RDkit_mix_complete.min()
max_value = y_cleaned_RDkit_mix_complete.max()
step_size = 1  # Adjust the step size for the x-axis
plt.xticks(np.arange(min_value, max_value + step_size, step_size))  # Set custom x-axis ticks
plt.yticks(np.arange(0, 11, 2))  # Set custom y-axis ticks
plt.show()


In [ ]:


# Assume:
# - X_numeric_RDkit_mix is your DataFrame containing numeric features.
# - y_cleaned_RDkit_mix is a pandas Series containing your target variable (Characteristic curvature, Cc).

# ---------------------------------------------------
# Step 0: Convert the target Series to a DataFrame
# ---------------------------------------------------
y_numeric_RDkit_mix = y_cleaned_RDkit_mix_complete.to_frame()  # Now it's 2D

# ---------------------------------------------------
# Step 1: Inspect the Data
# ---------------------------------------------------
print("Data Summary:")
print(y_numeric_RDkit_mix.describe())

# ---------------------------------------------------
# Step 2: Apply Yeo–Johnson Transformation to the target
# ---------------------------------------------------
pt = PowerTransformer(method='yeo-johnson')  # Automatically finds the best lambda.
y_yeo_transformed = pt.fit_transform(y_numeric_RDkit_mix)

# Convert back to DataFrame with a meaningful column name (e.g., 'Cc_transformed')
y_transformed_df = pd.DataFrame(y_yeo_transformed, columns=['Cc_transformed'])

print("\nYeo–Johnson Transformed Target:")
print(y_transformed_df.head())



In [ ]:
st = y_transformed_df

mu, sigma = st.mean(), st.std(ddof=1)
k_excesst = kurtosis(st, fisher=True, bias=False)
s_skewt   = skew(st, bias=False)

s_skewt

In [ ]:
df_kurtosis_p = y_numeric_RDkit_mix.kurtosis()
df_kurtosis_p

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Adjust the number of bins
sns.histplot(y_transformed_df['Cc_transformed'], 
             kde=True,  
             bins=15, 
             color = 'tab:blue',
             line_kws= {'color': 'blue'}
             )  # Adjust bins (e.g., 15 for larger steps)
plt.title('Distribution of Transformed Target Variable')
plt.xlabel('Cc_transformed')
plt.ylabel('Count')

# Customize the x-axis ticks
min_value = y_cleaned_RDkit_mix_complete.min()
max_value = y_cleaned_RDkit_mix_complete.max()
step_size = 1  # Adjust the step size for the x-axis
plt.xticks(np.arange(min_value, max_value + step_size, step_size))  # Set custom x-axis ticks
plt.yticks(np.arange(0, 11, 2))  # Set custom y-axis ticks
plt.show()

In [ ]:
df_kurtosis = y_transformed_df.kurtosis()
df_kurtosis

In [ ]:
# pip install numpy scipy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, skew, norm



mu, sigma = y_transformed_df.mean(), y_transformed_df.std(ddof=1)
k_excess = kurtosis(y_transformed_df, fisher=True, bias=False)
s_skew   = skew(y_transformed_df, bias=False)

plt.figure()
plt.hist(y_transformed_df, bins="auto", density=True,color='tab:blue',edgecolor='black', alpha=0.7)
xs = np.linspace(y_transformed_df.min(), y_transformed_df.max(), 400)
plt.plot(xs, norm.pdf(xs, mu, sigma), color = 'tab:blue')
plt.title(f"Kurtosis of Transformed target Vairable")
plt.xlabel("Cc_transformed"); plt.ylabel("Count")
ax = plt.gca()
ax.text(
    0.02, 0.95, "kurtosis score = -0.513",
    transform=ax.transAxes, ha="left", va="top",
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.6)
)
plt.show()


In [ ]:
# After-transform figure (somewhere earlier)
fig_t, ax_t = plt.subplots()
ax_t.hist(y_transformed_df, bins="auto", density=True, edgecolor='black', alpha=0.7)
ymin, ymax = ax_t.get_ylim()   # <-- save limits
plt.figure()
plt.hist(y_cleaned_RDkit_mix_complete, bins="auto", density=True,
         color='tab:blue', edgecolor='black', alpha=0.7)
xs = np.linspace(y_cleaned_RDkit_mix_complete.min(), y_cleaned_RDkit_mix_complete.max(), 400)
plt.plot(xs, norm.pdf(xs, mu, sigma), color='tab:blue')

plt.title("Kurtosis of Target Variable")
plt.xlabel("Cc"); plt.ylabel("Density")  # matches density=True
plt.ylim(ymin, ymax)                      # <-- same y-axis as transformed

ax = plt.gca()
ax.text(0.02, 0.95, "kurtosis score = -0.083",
        transform=ax.transAxes, ha="left", va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.6))
plt.show()


# Coding

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler

###########################################################################################
# 1) Define Models to Evaluate using SHAP-based Feature Ranking
###########################################################################################
models_complete = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Elastic Net Regression': ElasticNet(),
    'Decision Tree Regression': DecisionTreeRegressor(),
    'Random Forest Regression': RandomForestRegressor(),
    'Gradient Boosting Regression': GradientBoostingRegressor(),
    'XGBoost Regression': XGBRegressor()
}

# -----------------------------------------------------------
# 2) Split the dataset into training and testing sets (80/20)
# -----------------------------------------------------------
X_train_complete, X_test_complete, y_train_complete, y_test_complete = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=0.2, random_state=42
)

# (Optional) Ensure features are in a DataFrame and scaled.
X_train_complete = pd.DataFrame(X_train_complete, columns=pd.DataFrame(X_cleaned_RDkit_mix_complete1).columns)
X_test_complete = pd.DataFrame(X_test_complete, columns=pd.DataFrame(X_cleaned_RDkit_mix_complete1).columns)

n_features_complete = X_train_complete.shape[1]
print('Number of features:', n_features_complete)

# Dictionaries to store R² values for each model
model_train_r2_values_complete = {}
model_test_r2_values_complete = {}

###########################################################################################
# 3) Evaluate Models Using SHAP for Feature Ranking (Replacing RFE)
###########################################################################################
for model_name, model in models_complete.items():
    print(f"\nEvaluating {model_name} with SHAP-based feature selection...")
    
    # Fit the model on the full training set
    model.fit(X_train_complete, y_train_complete)
    
    # Choose an appropriate SHAP explainer:
    # For tree-based models, use TreeExplainer; for linear models, use LinearExplainer.
    if model_name in ['Decision Tree Regression', 'Random Forest Regression', 
                      'Gradient Boosting Regression', 'XGBoost Regression']:
        explainer = shap.TreeExplainer(model)
    elif model_name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 'Elastic Net Regression']:
        explainer = shap.LinearExplainer(model, X_train_complete, feature_perturbation="interventional")
    else:
        # Fallback to KernelExplainer (slower)
        explainer = shap.KernelExplainer(model.predict, X_train_complete.iloc[:100])  # use a subset for speed
    
    shap_values = explainer.shap_values(X_train_complete)
    
    # For linear models, shap_values might be a 2D array; for tree-based, also 2D.
    # Compute mean absolute SHAP value per feature.
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    
    # Rank features in descending order
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    sorted_features = X_train_complete.columns[sorted_indices]
    print("Feature ranking based on SHAP values:")
    print(sorted_features.tolist())
    
    train_r2_list = []
    test_r2_list = []
    
    # Loop over increasing numbers of top features based on SHAP ranking
    for i in range(1, n_features_complete + 1):
        top_features = sorted_features[:i]
        X_train_subset = X_train_complete[top_features]
        X_test_subset = X_test_complete[top_features]
        
        # Reinitialize and train the same model on the subset of features.
        # (If the model is stateful, reinitialize to ensure independence.)
        current_model = type(model)(**model.get_params())
        current_model.fit(X_train_subset, y_train_complete)
        y_train_pred = current_model.predict(X_train_subset)
        y_test_pred = current_model.predict(X_test_subset)
        
        train_r2 = r2_score(y_train_complete, y_train_pred)
        test_r2 = r2_score(y_test_complete, y_test_pred)
        
        train_r2_list.append(train_r2)
        test_r2_list.append(test_r2)
        
        print(f"Top {i} features: Train R² = {train_r2:.3f}, Test R² = {test_r2:.3f}")
    
    model_train_r2_values_complete[model_name] = train_r2_list
    model_test_r2_values_complete[model_name] = test_r2_list

    # Plot test R² as a function of number of features for this model
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, n_features_complete + 1), test_r2_list, marker='o', label=model_name)
    plt.title(f'R² vs. Number of Top Features (SHAP) - {model_name}')
    plt.xlabel('Number of Top Features')
    plt.ylabel('Test R² Score')
    plt.xticks(range(1, n_features_complete + 1))
    plt.grid(True)
    plt.legend()
    plt.show()

###########################################################################################
# 4) Summarize Results for All Models
###########################################################################################
print("\nFinal R² Scores at Optimal Feature Count:")
for model_name, test_r2_list in model_test_r2_values_complete.items():
    max_test_r2 = max(test_r2_list)
    max_idx = test_r2_list.index(max_test_r2)
    train_r2_at_max = model_train_r2_values_complete[model_name][max_idx]
    print(f"{model_name} - Max Test R²: {max_test_r2:.3f} (Train R²: {train_r2_at_max:.3f}) at {max_idx+1} features")

###########################################################################################
# 5) (Optional) Final Model Training on Full Data and SHAP Analysis
###########################################################################################
# Choose one model (for example, Gradient Boosting Regression) with optimal number of features
selected_model_name = 'Gradient Boosting Regression'
optimal_feature_count = model_test_r2_values_complete[selected_model_name].index(
    max(model_test_r2_values_complete[selected_model_name])
) + 1
optimal_features = sorted_features[:optimal_feature_count]  # Use the SHAP ranking from the last model loop

print(f"\nSelected Model: {selected_model_name} with top {optimal_feature_count} features.")

# Create optimal feature subsets
X_train_optimal = X_train_complete[optimal_features]
X_test_optimal = X_test_complete[optimal_features]

# Retrain the selected model on the full training data with optimal features
final_model = GradientBoostingRegressor(random_state=42)
final_model.fit(X_train_optimal, y_train_complete)

# Predict on training and testing sets
y_train_pred_final = final_model.predict(X_train_optimal)
y_test_pred_final = final_model.predict(X_test_optimal)

final_train_r2 = r2_score(y_train_complete, y_train_pred_final)
final_test_r2 = r2_score(y_test_complete, y_test_pred_final)
final_train_rmse = np.sqrt(mean_squared_error(y_train_complete, y_train_pred_final))
final_test_rmse = np.sqrt(mean_squared_error(y_test_complete, y_test_pred_final))

print(f"\nFinal Model Training Performance: Train R² = {final_train_r2:.3f}, RMSE = {final_train_rmse:.3f}")
print(f"Final Model Test Performance: Test R² = {final_test_r2:.3f}, RMSE = {final_test_rmse:.3f}")

# SHAP analysis on the final model using training data
explainer_final = shap.TreeExplainer(final_model)
shap_values_final = explainer_final.shap_values(X_train_optimal)

# SHAP Summary Beeswarm Plot
shap.summary_plot(shap_values_final, features=X_train_optimal, feature_names=X_train_optimal.columns)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.show()

# SHAP Summary Bar Plot
shap.summary_plot(shap_values_final, features=X_train_optimal, feature_names=X_train_optimal.columns, plot_type="bar")
plt.title("SHAP Bar Plot - Final Model")
plt.show()


In [ ]:
############################ optimization ############################

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = GradientBoostingRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3,2],
    'learning_rate': [0.01, 0.02,0.005],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    'random_state': [42]
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=GradientBoostingRegressor(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = GradientBoostingRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = GradientBoostingRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = GradientBoostingRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3,2],
    'learning_rate': [0.01, 0.02,0.005],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    'random_state': [42]
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=GradientBoostingRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = GradientBoostingRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=10,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = GradientBoostingRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = XGBRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=XGBRegressor(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = XGBRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = XGBRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = XGBRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=XGBRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = XGBRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=10,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = XGBRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = DecisionTreeRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=DecisionTreeRegressor(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = DecisionTreeRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = DecisionTreeRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = DecisionTreeRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=DecisionTreeRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = DecisionTreeRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=10,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = DecisionTreeRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix_complete.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = RandomForestRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = RandomForestRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = RandomForestRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import (train_test_split, GridSearchCV, 
                                     LeaveOneOut)
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# 1) Load Data (Ensure X has real column names)
# ---------------------------------------------------------------------

print("Full dataset shape:",X_cleaned_RDkit_mix_complete1.shape)
print("y shape:",y_cleaned_RDkit_mix_complete.shape)

# Train/Hold-out split (e.g., 80:20) for final unbiased test
test_size_ratio = 0.2  
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_cleaned_RDkit_mix_complete1, y_cleaned_RDkit_mix_complete, test_size=test_size_ratio, random_state=42
)

print(f"Training set shape: {X_train.shape}, Hold-out set shape: {X_holdout.shape}")

# ---------------------------------------------------------------------
# 2) Quick Model for SHAP Ranking on Training Set
# ---------------------------------------------------------------------
quick_model = RandomForestRegressor(random_state=42)
quick_model.fit(X_train, y_train)

explainer_quick = shap.TreeExplainer(quick_model)
shap_values_quick = explainer_quick.shap_values(X_train)

# Mean absolute SHAP
mean_abs_shap = np.abs(shap_values_quick).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap)[::-1]  # descending order

# ---------------------------------------------------------------------
# 3) Define Hyperparameter Grid & LOOCV
# ---------------------------------------------------------------------
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}


loo = LeaveOneOut()
max_features = X_train.shape[1]

results = []

# ---------------------------------------------------------------------
# 4) Loop over #Features (from SHAP ranking), Tune with LOOCV
# ---------------------------------------------------------------------

for n_features in range(1, max_features + 1):
    topn_idx     = sorted_indices[:n_features]
    X_train_top  = X_train.iloc[:, topn_idx]

    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(random_state=42),
        param_grid=param_grid,
        cv=10,
        scoring='neg_mean_squared_error',
        return_train_score=True,   # <-- so cv_results_ has every split score
        n_jobs=-1,
        verbose=1,
        error_score='raise'
    )

    try:
        grid_search.fit(X_train_top, y_train)
    except ValueError as e:
        print(f"Skipping n_features={n_features}, error: {e}")
        continue

    # --- 1) CV RMSE (mean ± std) from the grid_search splits ---
    cv_res    = grid_search.cv_results_
    best_idx  = grid_search.best_index_
    neg_mses  = np.array([
        cv_res[f"split{i}_test_score"][best_idx] for i in range(5)
    ])
    rmses     = np.sqrt(-neg_mses)
    cv_mean_rmse = rmses.mean()
    cv_std_rmse  = rmses.std()

    # --- 2) CV R² (mean ± std) via cross_val_score on best_params ---
    best_params = grid_search.best_params_
    model_for_r2 = RandomForestRegressor(**best_params)
    r2_scores    = cross_val_score(
        model_for_r2,
        X_train_top,
        y_train,
        cv=10,
        scoring='r2',
        n_jobs=-1
    )
    cv_mean_r2 = r2_scores.mean()
    cv_std_r2  = r2_scores.std()

    # --- 3) Retrain on full train & evaluate on holdout ---
    best_model = grid_search.best_estimator_
    best_model.fit(X_train_top, y_train)
    y_train_pred   = best_model.predict(X_train_top)
    X_holdout_top  = X_holdout.iloc[:, topn_idx]
    y_test_pred    = best_model.predict(X_holdout_top)

    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_r2    = r2_score(y_holdout, y_test_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_holdout, y_test_pred))
    test_mae   = mean_absolute_error(y_holdout, y_test_pred)

    # --- 4) Save everything ---
    results.append({
        'Num Features':  n_features,
        'Best Params':   best_params,
        'CV Mean RMSE':  cv_mean_rmse,
        'CV Std RMSE':   cv_std_rmse,
        'CV Mean R2':    cv_mean_r2,
        'CV Std R2':     cv_std_r2,
        'Train R2':      train_r2,
        'Train RMSE':    train_rmse,
        'Test R2':       test_r2,
        'Test RMSE':     test_rmse,
        'Test MAE':      test_mae
    })

    print(
        f"[n={n_features}] "
        f"CV RMSE: {cv_mean_rmse:.3f}±{cv_std_rmse:.3f}, "
        f"CV R2: {cv_mean_r2:.3f}±{cv_std_r2:.3f}, "
        f"Train R2: {train_r2:.3f}, Test R2: {test_r2:.3f}"
    )

results_df = pd.DataFrame(results)
print("\nAll Results:\n", results_df)


# ---------------------------------------------------------------------
# 5) Pick Best Model by Test R²
# ---------------------------------------------------------------------
best_idx = results_df['Test R2'].idxmax()
best_row = results_df.loc[best_idx]
print("\nBest model configuration:\n", best_row)

# Extract values for plotting
best_train_r2 = best_row['Train R2']
best_test_r2 = best_row['Test R2']
best_train_rmse = best_row['Train RMSE']
best_test_rmse = best_row['Test RMSE']
best_n_features = best_row['Num Features']
best_test_mae = best_row['Test MAE']

best_n_features = best_row['Num Features']
best_params = best_row['Best Params']

# Re-create final best model for SHAP & plots
topn_idx = sorted_indices[:best_n_features]

X_train_best = X_train.iloc[:, topn_idx]
X_holdout_best = X_holdout.iloc[:, topn_idx]

best_final_model = RandomForestRegressor(**best_params)
best_final_model.fit(X_train_best, y_train)

y_train_pred_best = best_final_model.predict(X_train_best)
y_test_pred_best = best_final_model.predict(X_holdout_best)

# ---------------------------------------------------------------------
# 6) Combined Regression Plot (Train + Test)
# ---------------------------------------------------------------------
plt.figure(figsize=(8, 6))

# Plot train predictions
plt.scatter(
    y_train, 
    y_train_pred_best, 
    alpha=0.7, 
    label=f"Train Data\nRMSE: {best_train_rmse:.2f}, R²: {best_train_r2:.2f}",
    color="blue"
)
# Plot test predictions
plt.scatter(
    y_holdout,
    y_test_pred_best,
    alpha=0.7,
    label=f"Test Data\nRMSE: {best_test_rmse:.2f}, R²: {best_test_r2:.2f}, MAE: {best_test_mae:.2f}",
    color="orange",
)

# Common axis limits
all_true = np.concatenate([y_train, y_holdout])
all_pred = np.concatenate([y_train_pred_best, y_test_pred_best])

lims = [min(all_true.min(), all_pred.min()),
        max(all_true.max(), all_pred.max())]

plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)

# Set x-axis and y-axis ticks with step size of 1
min_val = int(min(y_train.min(), y_holdout.min()))
max_val = int(max(y_train.max(), y_holdout.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))

plt.title("Accutual vs Predicted Values (Final Model)")
plt.xlabel("Accual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.savefig("combined_parity_plot.png")
plt.show()

# ---------------------------------------------------------------------
# 7) SHAP Analysis on Final Model
# ---------------------------------------------------------------------
explainer_final = shap.TreeExplainer(best_final_model)
shap_values_final = explainer_final.shap_values(X_train_best)

# (a) SHAP Beeswarm Plot (default summary_plot)
# Pass 'features=X_train_best' and 'feature_names=X_train_best.columns'
# to ensure real column names appear.
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    show=False
)
plt.title("SHAP Beeswarm - Positive & Negative Impact")
plt.savefig("shap_summary_beeswarm.png")
plt.show()

# (b) SHAP Bar Plot => global mean(|SHAP|)
shap.summary_plot(
    shap_values_final,
    features=X_train_best,
    feature_names=X_train_best.columns,
    plot_type="bar",
    show=False
)
plt.title("SHAP Bar Plot (Mean Absolute Impact)")
plt.savefig("shap_summary_bar.png")
plt.show()

# (c) Optional: Waterfall Plot => local explanation for 1 sample
sample_index = 0
base_value = explainer_final.expected_value  # baseline (avg model output)
sample_shap = shap_values_final[sample_index, :]  # shap for that sample
try:
    # For newer SHAP versions:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0.png")
    plt.show()
except AttributeError:
    # For older SHAP versions:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_best.iloc[sample_index, :],
            feature_names=X_train_best.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_legacy.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot.png")
print("  - shap_summary_beeswarm.png (positive/negative impact distribution)")
print("  - shap_summary_bar.png      (global importance ranking)")
print("  - shap_waterfall_sample0.png (local explanation example for sample=0)")


In [ ]:
###################### Nest #####################################

In [ ]:
################# STD ###################################################

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3,2],
    'learning_rate': [0.01, 0.02,0.005],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    'random_state': [42]
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3,2],
    'learning_rate': [0.01, 0.02,0.005],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    'random_state': [42]
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0,
            error_score= 'raise'
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB_1, y_XGB_1=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB_1, y_XGB_1, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB_1 = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB_1 = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB_1.shape)
print("y shape:", y_XGB_1.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}
max_features = X_XGB_1.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB_1, y_XGB_1), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB_1.iloc[train_index]
    y_train_outer_XGB = y_XGB_1[train_index]
    X_test_outer_XGB  = X_XGB_1.iloc[test_index]
    y_test_outer_XGB  = y_XGB_1[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB_1, y_XGB_1, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB_1.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB_1.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
############### STD + YEO (TARGET) ###########################

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3],
    'learning_rate': [0.01, 0.02],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3],
    'learning_rate': [0.01, 0.02],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0,
            error_score='raise'  # Raise error if any occurs during fitting
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0,
            error_score='raise'  # Raise error if any occurs during fitting
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_cleaned_RDkit_mix_complete1)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
################## RBS #############################################

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler, MaxAbsScaler, QuantileTransformer, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
# ------------------------------
# Step 1: Load Data and Define X and y
# ------------------------------



# ------------------------------
# Step 2: Remove Constant Features (Variance = 0)
# ------------------------------
def remove_constant_features(df, threshold=0.0):
    """
    Remove features that are constant (have only one unique value).
    """
    constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
    df_reduced = df.drop(columns=constant_cols)
    return df_reduced, constant_cols

X_no_const, dropped_const = remove_constant_features(X_cleaned_RDkit_mix_complete, threshold=0.0)
print("\nDropped constant features:", dropped_const)
print("Shape after constant removal:", X_no_const.shape)


# ------------------------------
# Step 4: Group-wise Correlation Elimination
# ------------------------------
def remove_highly_correlated_features(df, threshold=0.8):
    """
    Remove features with pairwise absolute correlation greater than the threshold.
    Returns the reduced DataFrame and list of dropped columns.
    """
    # Compute absolute correlation matrix of all numeric features.
    corr_matrix = df.corr().abs()
    # Create upper triangle mask
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    # Find columns to drop
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    df_reduced = df.drop(columns=to_drop)
    return df_reduced, to_drop

X_global_reduced, dropped_global = remove_highly_correlated_features(X_no_const, threshold=0.8)
print("\nGlobal correlation elimination:")
print("Dropped columns:", dropped_global)
print("Shape after global elimination:", X_global_reduced.shape)



# ------------------------------
# Step 5: Scaling the Features
# ------------------------------

scaler_global = Pipeline([
    ('robust', RobustScaler(quantile_range=(10, 90))),
    
])

X_global_scaled_RBS = pd.DataFrame(scaler_global.fit_transform(X_global_reduced), 
                               columns=X_global_reduced.columns, 
                               index=X_global_reduced.index)
print("\nGlobal features scaled.")

# ------------------------------
# Step 6: Plot Correlation Heatmaps for the Scaled Datasets
# ------------------------------

# Heatmap for Global Reduced & Scaled Features
corr_global_scaled = X_global_scaled_RBS.corr()
plt.imshow(corr_global_scaled, interpolation='none')
plt.colorbar()
# Plot the correlation heatmap with increased annotation font size
plt.figure(figsize=(120, 100))  # Set figure size for better readability
sns.heatmap(corr_global_scaled, 
            annot=True, 
            annot_kws={"size": 40},  # Increase annotation font size
            cmap='coolwarm', 
            fmt=".2f", 
            linewidths=0.5, 
            cbar=True, 
            square=True)

# Increase the font size of the x and y axis tick labels
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.xticks(range(len(corr_global_scaled.columns)), corr_global_scaled.columns, rotation=90)
plt.yticks(range(len(corr_global_scaled.columns)), corr_global_scaled.columns)
plt.title('Global Reduced & Scaled Features Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3,2],
    'learning_rate': [0.01, 0.02,0.005],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3,2],
    'learning_rate': [0.01, 0.02,0.005],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
        'n_estimators': [900,1000,1100,1200,1300,1500],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0,
            error_score='raise'
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB = {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X, y=None, groups=None):
        return self.kf.get_n_splits(X, y, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_cleaned_RDkit_mix_complete is your target.
X_XGB = pd.DataFrame(X_global_scaled_RBS)
y_XGB = np.array(y_cleaned_RDkit_mix_complete)

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for XGBRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        gbr_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=gbr_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# Evaluate performance
train_r2_final = r2_score(y_train_full, y_train_pred_final)
train_rmse_final = np.sqrt(mean_squared_error(y_train_full, y_train_pred_final))
train_mae_final = mean_absolute_error(y_train_full, y_train_pred_final)
holdout_r2_final = r2_score(y_holdout, y_holdout_pred_final)
holdout_rmse_final = np.sqrt(mean_squared_error(y_holdout, y_holdout_pred_final))
houldout_mae_final = mean_absolute_error(y_holdout, y_holdout_pred_final)



print(f"\nFinal Model Training Performance: Train R2 = {train_r2_final:.3f}, Train RMSE = {train_rmse_final:.3f}, Train MAE = {train_mae_final:.3f}")
print(f"Final Model Hold-out Performance: Holdout R2 = {holdout_r2_final:.3f}, Holdout RMSE = {holdout_rmse_final:.3f}, Holdout MAE = {houldout_mae_final:.3f}")


# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full, y_train_pred_final, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final:.2f}, R²: {train_r2_final:.2f}, MAE: {train_mae_final:.2f}")
plt.scatter(y_holdout, y_holdout_pred_final, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final:.2f}, R²: {holdout_r2_final:.2f}, MAE: {houldout_mae_final:.2f}")
lims = [min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()),
        max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_XGB.min(), y_train_pred_final.min(), y_holdout_pred_final.min()))
max_val = int(max(y_XGB.max(), y_train_pred_final.max(), y_holdout_pred_final.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")
print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")


In [ ]:
###################################################################################################

In [ ]:
####################################################  Yeo-Johnson Transformation (Target) + Robustness (Feature)  ###########################################################################

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, RobustScaler, MaxAbsScaler, QuantileTransformer, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
# ------------------------------
# Step 1: Load Data and Define X and y
# ------------------------------



# ------------------------------
# Step 2: Remove Constant Features (Variance = 0)
# ------------------------------
def remove_constant_features(df, threshold=0.0):
    """
    Remove features that are constant (have only one unique value).
    """
    constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
    df_reduced = df.drop(columns=constant_cols)
    return df_reduced, constant_cols

X_no_const, dropped_const = remove_constant_features(X_cleaned_RDkit_mix_complete, threshold=0.0)
print("\nDropped constant features:", dropped_const)
print("Shape after constant removal:", X_no_const.shape)


# ------------------------------
# Step 4: Group-wise Correlation Elimination
# ------------------------------
def remove_highly_correlated_features(df, threshold=0.8):
    """
    Remove features with pairwise absolute correlation greater than the threshold.
    Returns the reduced DataFrame and list of dropped columns.
    """
    # Compute absolute correlation matrix of all numeric features.
    corr_matrix = df.corr().abs()
    # Create upper triangle mask
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    # Find columns to drop
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    df_reduced = df.drop(columns=to_drop)
    return df_reduced, to_drop

X_global_reduced, dropped_global = remove_highly_correlated_features(X_no_const, threshold=0.8)
print("\nGlobal correlation elimination:")
print("Dropped columns:", dropped_global)
print("Shape after global elimination:", X_global_reduced.shape)



# ------------------------------
# Step 5: Scaling the Features
# ------------------------------

scaler_global = Pipeline([
    ('robust', RobustScaler(quantile_range=(10, 90))),
    ('minmax', MinMaxScaler())
])

X_global_scaled_RBS_Y = pd.DataFrame(scaler_global.fit_transform(X_global_reduced), 
                               columns=X_global_reduced.columns, 
                               index=X_global_reduced.index)
print("\nGlobal features scaled.")

# ------------------------------
# Step 6: Plot Correlation Heatmaps for the Scaled Datasets
# ------------------------------

# Heatmap for Global Reduced & Scaled Features
corr_global_scaled = X_global_scaled_RBS_Y.corr()
plt.imshow(corr_global_scaled, interpolation='none')
plt.colorbar()
# Plot the correlation heatmap with increased annotation font size
plt.figure(figsize=(120, 100))  # Set figure size for better readability
sns.heatmap(corr_global_scaled, 
            annot=True, 
            annot_kws={"size": 40},  # Increase annotation font size
            cmap='coolwarm', 
            fmt=".2f", 
            linewidths=0.5, 
            cbar=True, 
            square=True)

# Increase the font size of the x and y axis tick labels
plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.xticks(range(len(corr_global_scaled.columns)), corr_global_scaled.columns, rotation=90)
plt.yticks(range(len(corr_global_scaled.columns)), corr_global_scaled.columns)
plt.title('Global Reduced & Scaled Features Correlation Heatmap')
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3],
    'learning_rate': [0.01, 0.02],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [800,900,1000,1100,1200,1300],
    'max_depth': [3],
    'learning_rate': [0.01, 0.02],
    'subsample': [1.0, 0.8, 0.6],
    'min_samples_leaf': [5,7,10],
    'max_features': ['sqrt', 'log2', 0.5],
    'random_state': [42]
}

max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = GradientBoostingRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = GradientBoostingRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = GradientBoostingRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = GradientBoostingRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
        'n_estimators': [900,1000,1100,1200],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
#Goal

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
        'n_estimators': [900,1000,1100,1200],
        'max_depth': [3],
        'learning_rate': [0.01, 0.02],
        'subsample': [1.0, 0.8, 0.6],
        'reg_lambda': [1,5,10],
        'reg_alpha': [0, 0.1,1],
        'gamma': [0, 0.1, 1],
        'min_child_weight': [5, 10],
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = XGBRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = XGBRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = XGBRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
        'Outer MAE (orig)': outer_mae_orig.item(),
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = XGBRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
outer_results_df_XGB

In [ ]:
plt.figure(figsize=(8, 6))
plt.rcParams["font.family"] = "Times New Roman"
# Scatter plots
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")

# Axis limits
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]

# Perfect prediction and MAE bounds
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
mae = holdout_mae_final_orig  # or choose to show average of train and holdout MAEs
plt.plot(lims, [y + mae for y in lims], 'k--', label=f'+MAE ({mae:.2f})')
plt.plot(lims, [y - mae for y in lims], 'k--', label=f'-MAE ({mae:.2f})')


# Plot configuration
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(lims))
max_val = int(max(lims))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout_with_MAE_lines.png")
plt.show()

In [ ]:
# --- right after you compute y_pred_outer_XGB and before appending to outer_results_XGB ---
# Inverse-transform the outer test preds/targets to original scale
col_name = 'Cc'  # same name you used with pt
y_pred_outer_orig = pt.inverse_transform(pd.DataFrame(y_pred_outer_XGB.reshape(-1,1), columns=[col_name]))
y_test_outer_orig = pt.inverse_transform(pd.DataFrame(y_test_outer_XGB.reshape(-1,1), columns=[col_name]))

outer_mae_orig = mean_absolute_error(y_test_outer_orig, y_pred_outer_orig)

outer_results_XGB.append({
    'Fold': outer_fold,
    'Num Features': best_n_features_XGB,
    'Best Params': best_params_XGB,
    'Outer R2': outer_r2_XGB,
    'Outer RMSE': outer_rmse_XGB,
    'Top Indices': best_top_indices_XGB,
    'Outer Train Indices': train_index,
    'Outer Test Indices': test_index,
    'Outer MAE': outer_mae_XGB,                # transformed scale
    'Outer MAE (orig)': outer_mae_orig.item(), # original scale
})


In [ ]:
# =========================================
#  Compute 95% CI for MAE (over folds)
#  Choose the column: use 'Outer MAE (orig)' if available, else 'Outer MAE'
# =========================================
mae_col = 'Outer MAE (orig)' if 'Outer MAE (orig)' in outer_results_df_XGB.columns else 'Outer MAE'
maes = outer_results_df_XGB[mae_col].astype(float).to_numpy()
n = maes.size
mae_mean = maes.mean()
mae_se = maes.std(ddof=1) / np.sqrt(n)

# Try a Student-t critical value; fall back to normal if SciPy isn't present
try:
    from scipy.stats import t
    crit = t.ppf(0.975, df=n-1)
except Exception:
    crit = 1.96  # normal approx

mae_ci = crit * mae_se
ci_lo, ci_hi = mae_mean - mae_ci, mae_mean + mae_ci

print(f"{mae_col}  Mean: {mae_mean:.3f},  95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]")


In [ ]:
plt.figure(figsize=(8, 6))
plt.rcParams["font.family"] = "Times New Roman"

# Scatter plots
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")

# Axis limits
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(),
            y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(),
            y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]

# Perfect prediction
plt.plot(lims, lims, 'r--', label='Perfect Prediction')

# --- Mean MAE lines (over folds) ---
plt.plot(lims, [y + mae_mean for y in lims], 'k--', label=f'+Mean MAE ({mae_mean:.2f})')
plt.plot(lims, [y - mae_mean for y in lims], 'k--', label=f'-Mean MAE ({mae_mean:.2f})')

# --- 95% CI shaded bands around y=x (parallel offset by mean±CI) ---
# below diagonal (pred = actual - offset)
plt.fill_between(
    lims,
    [y - (mae_mean + mae_ci) for y in lims],
    [y - (mae_mean - mae_ci) for y in lims],
    alpha=0.12, label=f'95% CI of MAE (n={n})'
)
# above diagonal (pred = actual + offset)
plt.fill_between(
    lims,
    [y + (mae_mean - mae_ci) for y in lims],
    [y + (mae_mean + mae_ci) for y in lims],
    alpha=0.12
)

# (Optional) keep your single hold-out MAE lines if you still want them:
# plt.plot(lims, [y + holdout_mae_final_orig for y in lims], 'gray', linestyle=':', label=f'+Hold-out MAE ({holdout_mae_final_orig:.2f})')
# plt.plot(lims, [y - holdout_mae_final_orig for y in lims], 'gray', linestyle=':', label=f'-Hold-out MAE ({holdout_mae_final_orig:.2f})')

# Plot configuration
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(lims))
max_val = int(max(lims))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model) with MAE 95% CI over Folds")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_with_MAE_CI.png", dpi=300)
plt.show()


In [ ]:
# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()


In [ ]:
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.rcParams["font.family"] = "Times New Roman"

# Scatter plots
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")

# Axis limits
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(),
            y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(),
            y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]

# Perfect prediction and MAE bounds
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
mae = holdout_mae_final_orig  # or average MAE if you prefer
plt.plot(lims, [y + mae for y in lims], 'k--', label=f'+MAE ({mae:.2f})')
plt.plot(lims, [y - mae for y in lims], 'k--', label=f'-MAE ({mae:.2f})')

# =========================================================
# 1) R² 95% CI (prefer outer folds; else Fisher-z on holdout)
# =========================================================
r2_text = None
if 'outer_results_df_XGB' in globals() and not outer_results_df_XGB.empty and 'Outer R2' in outer_results_df_XGB:
    r2s = outer_results_df_XGB['Outer R2'].astype(float).to_numpy()
    n_r2 = r2s.size
    r2_mean = r2s.mean()
    r2_se = r2s.std(ddof=1) / np.sqrt(n_r2)
    try:
        from scipy.stats import t
        crit = t.ppf(0.975, df=n_r2-1)
    except Exception:
        crit = 1.96
    r2_lo, r2_hi = r2_mean - crit*r2_se, r2_mean + crit*r2_se
    r2_text = f"Outer R²: {r2_mean:.3f} [{r2_lo:.3f}, {r2_hi:.3f}]"
else:
    # Fisher-z CI from hold-out correlation
    y_true = np.ravel(y_holdout_orig)
    y_pred = np.ravel(y_holdout_pred_final_orig)
    r = np.corrcoef(y_true, y_pred)[0, 1]
    n = y_true.size
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))
    se = 1/np.sqrt(max(n-3, 1))
    z_lo, z_hi = z - 1.96*se, z + 1.96*se
    r_lo, r_hi = np.tanh(z_lo), np.tanh(z_hi)
    r2_lo, r2_hi = r_lo**2, r_hi**2
    r2_text = f"Hold-out R²: {r**2:.3f} [{r2_lo:.3f}, {r2_hi:.3f}]"

    

# =========================================================
# 2) Calibration line with 95% bootstrap CI ribbon
#    Fit y_pred = a + b * y_true on HOLD-OUT set
# =========================================================
x = np.ravel(y_holdout_orig)               # actual
y = np.ravel(y_holdout_pred_final_orig)    # predicted
# Fit once for median line
b_hat, a_hat = np.polyfit(x, y, deg=1)

# Bootstrap slope/intercept
rng = np.random.default_rng(42)
B = 2000
n = x.size
bs_coefs = np.empty((B, 2))
for i in range(B):
    idx = rng.integers(0, n, size=n)
    bb, aa = np.polyfit(x[idx], y[idx], deg=1)
    bs_coefs[i] = [bb, aa]

# Build grid over displayed range and compute percentile bands
xgrid = np.linspace(lims[0], lims[1], 200)
ygrid_samples = bs_coefs[:, 0:1] * xgrid[None, :] + bs_coefs[:, 1:2]  # (B, 200)
lo = np.percentile(ygrid_samples, 2.5, axis=0)
hi = np.percentile(ygrid_samples, 97.5, axis=0)
med = b_hat * xgrid + a_hat
# =========================================================
# 2b) Calibration line with 95% bootstrap CI ribbon
#     Fit y_pred = a + b * y_true on TRAINING set
# =========================================================
x_tr = np.ravel(y_train_full_orig)           # actual (train)
y_tr = np.ravel(y_train_pred_final_orig)     # predicted (train)

# Fit once for median line (train)
b_hat_tr, a_hat_tr = np.polyfit(x_tr, y_tr, deg=1)

# Bootstrap slope/intercept (train)
rng_tr = np.random.default_rng(43)
B_tr = 2000
n_tr = x_tr.size
bs_coefs_tr = np.empty((B_tr, 2))
for i in range(B_tr):
    idx = rng_tr.integers(0, n_tr, size=n_tr)
    bb_tr, aa_tr = np.polyfit(x_tr[idx], y_tr[idx], deg=1)
    bs_coefs_tr[i] = [bb_tr, aa_tr]

# Use the same grid as before for comparability
ygrid_samples_tr = bs_coefs_tr[:, 0:1] * xgrid[None, :] + bs_coefs_tr[:, 1:2]
lo_tr = np.percentile(ygrid_samples_tr, 2.5, axis=0)
hi_tr = np.percentile(ygrid_samples_tr, 97.5, axis=0)
med_tr = b_hat_tr * xgrid + a_hat_tr

# Draw TRAIN calibration line and ribbon
plt.plot(xgrid, med_tr, linestyle='-', linewidth=2, color='tab:orange',
         label='Train calibration line')
plt.fill_between(xgrid, lo_tr, hi_tr, alpha=0.15, color='tab:orange',
                 label='Train 95% CI')

# Draw calibration line and ribbon
plt.plot(xgrid, med, linestyle='-', linewidth=2, label='Calibration line')
plt.fill_between(xgrid, lo, hi, alpha=0.15, label='Calibration 95% CI')

# Plot configuration
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(lims))
max_val = int(max(lims))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.legend()


# Annotate R² CI on the figure
#

plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout_with_MAE_R2_CI_and_calibration.png", dpi=300)
plt.show()


In [ ]:
# --- global sizing (optional) ---
import matplotlib as mpl
mpl.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "savefig.dpi": 300,
})

from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import numpy as np

# mean(|SHAP|) ranking you already computed
shap_vals = shap_values_final_XGB if isinstance(shap_values_final_XGB, np.ndarray) else shap_values_final_XGB.values
rank = np.argsort(np.abs(shap_vals).mean(axis=0))[::-1]
topk = min(8, len(rank))
top_features = list(X_train_full_optimal.columns[rank[:topk]])

fig, axes = plt.subplots(nrows=topk, ncols=1, figsize=(6, 3.0*topk))
axes = np.atleast_1d(axes).ravel()   # <- make sure it's a flat list of Axes

for ax, feat in zip(axes, top_features):
    disp = PartialDependenceDisplay.from_estimator(
        final_model_full_XGB, X_train_full_optimal,
        features=[feat], kind="average", grid_resolution=50, ax=ax
    )
    # Set title on the axes actually used by sklearn
    used_axes = np.ravel(getattr(disp, "axes_", np.array([ax])))
    used_axes[0].set_title(f"PDP: {feat}", pad=6)   # per-plot title
    ax.grid(True)

fig.tight_layout()
fig.savefig("pdp_topk_by_shap.png", bbox_inches="tight")
plt.show()


In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import numpy as np

# mean(|SHAP|) ranking you already computed
shap_vals = shap_values_final_XGB if isinstance(shap_values_final_XGB, np.ndarray) else shap_values_final_XGB.values
rank = np.argsort(np.abs(shap_vals).mean(axis=0))[::-1]
topk = 8
top_features = X_train_full_optimal.columns[rank[:topk]]

fig, axes = plt.subplots(nrows=topk, ncols=1, figsize=(6, 3.0*topk))
for ax, feat in zip(axes, top_features):
    PartialDependenceDisplay.from_estimator(
        final_model_full_XGB, X_train_full_optimal,
        features=[feat], kind="average", grid_resolution=50, ax=ax
    )
        # Set axis label font sizes
    ax.set_xlabel(ax.get_xlabel(), fontsize=140)
    ax.set_ylabel(ax.get_ylabel(), fontsize=140)
    ax.tick_params(axis='both', labelsize=120)  # Set tick label font size
    ax.set_title(f"PDP: {feat}")
    ax.grid(True)




plt.tight_layout()
plt.savefig("pdp_topk_by_shap.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# SHAP values (array form)
shap_values_array = shap_values_final_XGB.values if hasattr(shap_values_final_XGB, "values") else shap_values_final_XGB

# Mean(|SHAP|) across samples
mean_abs_shap = np.abs(shap_values_array).mean(axis=0)

# DataFrame for sorting
shap_df = pd.DataFrame({
    "feature": X_train_full_optimal.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False).head(20)   # top 20

# Plot vertical bar chart
plt.figure(figsize=(10, 6))
plt.bar(shap_df["feature"], shap_df["mean_abs_shap"], color="skyblue")
plt.xticks(rotation=90, fontsize=10)
plt.ylabel("mean(|SHAP value|)", fontsize=12)
plt.title("Top 20 Features by mean(|SHAP|)", fontsize=14)

plt.tight_layout()
plt.savefig("shap_summary_bar_vertical_top20.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import shap
import numpy as np
import pandas as pd

# Compute mean(|SHAP|) per feature
shap_values_array = shap_values_final_XGB.values if hasattr(shap_values_final_XGB, "values") else shap_values_final_XGB
mean_abs_shap = np.abs(shap_values_array).mean(axis=0)

# Put into a DataFrame for easy sorting
shap_df = pd.DataFrame({
    "feature": X_train_full_optimal.columns,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

# Vertical bar plot
plt.figure(figsize=(10, 6))
plt.bar(shap_df["feature"], shap_df["mean_abs_shap"], color="skyblue")
plt.xticks(rotation=90, fontsize=10)
plt.ylabel("mean(|SHAP value|)")
plt.title("SHAP Bar Plot (Final Model) - Vertical")

plt.tight_layout()
plt.savefig("shap_summary_bar_nested_XGB_holdout_vertical.png", dpi=600)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for DecisionTreeRegressor
# ============================================================
param_grid_XGB =  {
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'random_state': [42]
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = DecisionTreeRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = DecisionTreeRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = DecisionTreeRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = DecisionTreeRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=5, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for RandomForestRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=5, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=5, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import shap

# ============================================================
# Custom LoggingKFold for inner CV: prints and logs splits
# ============================================================
class LoggingKFold:
    def __init__(self, n_splits=10, shuffle=True, random_state=42):
        self.kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
        self.splits_log = []  # List to store splits

    def split(self, X_XGB, y_XGB=None, groups=None):
        for train_index, test_index in self.kf.split(X_XGB, y_XGB, groups):
            self.splits_log.append((train_index, test_index))
            print("  Inner CV Split: Train indices:", train_index, " Test indices:", test_index)
            yield train_index, test_index

    def get_n_splits(self, X_XGB, y_XGB=None, groups=None):
        return self.kf.get_n_splits(X_XGB, y_XGB, groups)

# ============================================================
# 1) Load Data
# ============================================================
# Assume X_cleaned_RDkit_mix_complete is your DataFrame with real feature names,
# and y_transformed_df is your target after applying the Yeo–Johnson transformation.
X_XGB = pd.DataFrame(X_global_scaled_RBS_Y)
y_XGB = np.array(y_transformed_df).ravel()  # Transformed target, flattened to 1D

print("Full dataset shape:", X_XGB.shape)
print("y shape:", y_XGB.shape)

# ============================================================
# 2) Define Parameter Grid for RandomForestRegressor
# ============================================================
param_grid_XGB =  {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'random_state': [42]
}
max_features = X_XGB.shape[1]

# ============================================================
# 3) Outer Cross-Validation Setup (Nested CV)
# ============================================================
outer_kf = KFold(n_splits=10, shuffle=True, random_state=42)
outer_results_XGB = []

for outer_fold, (train_index, test_index) in enumerate(outer_kf.split(X_XGB, y_XGB), start=1):
    print(f"\n=== Outer Fold {outer_fold} ===")
    print("Outer Train indices:", train_index)
    print("Outer Test indices: ", test_index)
    
    # Split outer train/test data
    X_train_outer_XGB = X_XGB.iloc[train_index]
    y_train_outer_XGB = y_XGB[train_index]
    X_test_outer_XGB  = X_XGB.iloc[test_index]
    y_test_outer_XGB  = y_XGB[test_index]
    
    # --------------------------------------------------
    # a) Feature Ranking on Outer Training Data using a Quick Model
    # --------------------------------------------------
    quick_model_XGB = RandomForestRegressor(random_state=42)
    quick_model_XGB.fit(X_train_outer_XGB, y_train_outer_XGB)
    explainer_quick = shap.TreeExplainer(quick_model_XGB)
    shap_values_quick_XGB = explainer_quick.shap_values(X_train_outer_XGB)
    
    # Calculate mean absolute SHAP value per feature and sort in descending order
    mean_abs_shap = np.abs(shap_values_quick_XGB).mean(axis=0)
    sorted_indices = np.argsort(mean_abs_shap)[::-1]
    
    # --------------------------------------------------
    # b) Inner CV for Hyperparameter Tuning & Feature Selection
    # --------------------------------------------------
    inner_kf = LoggingKFold(n_splits=10, shuffle=True, random_state=42)
    inner_results_XGB = []
    
    for n_features in range(1, max_features + 1):
        topn_idx = sorted_indices[:n_features]
        X_train_top_XGB = X_train_outer_XGB.iloc[:, topn_idx]
        
        # GridSearchCV on inner CV using selected features
        model_XGB = RandomForestRegressor(random_state=42)
        grid_search = GridSearchCV(
            estimator=model_XGB,
            param_grid=param_grid_XGB,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1,
            verbose=0
        )
        
        try:
            grid_search.fit(X_train_top_XGB, y_train_outer_XGB)
        except ValueError as e:
            print(f"Skipping n_features={n_features} due to error: {e}")
            continue
        
        best_model_inner_XGB = grid_search.best_estimator_
        best_params_inner_XGB = grid_search.best_params_
        inner_mse_XGB = -grid_search.best_score_
        inner_rmse_XGB = np.sqrt(inner_mse_XGB)
        inner_r2_XGB = r2_score(y_train_outer_XGB, best_model_inner_XGB.predict(X_train_top_XGB))
        
        inner_results_XGB.append({
            'Num Features': n_features,
            'Best Params': best_params_inner_XGB,
            'Inner RMSE': inner_rmse_XGB,
            'Inner R2': inner_r2_XGB,
            'Top Indices': topn_idx,
            'Best Model': best_model_inner_XGB
        })
        
        print(f"Outer Fold {outer_fold} | n_features={n_features}: Inner R2={inner_r2_XGB:.3f}, Inner RMSE={inner_rmse_XGB:.3f}")
    
    # --------------------------------------------------
    # c) Choose Best Inner Configuration using tie-breaker:
    #    - Filter candidates with maximum Inner R2 (using a tolerance),
    #      then select the candidate with the lowest RMSE among them.
    # --------------------------------------------------
    if inner_results_XGB:
        epsilon = 1e-6  # Tolerance for floating point comparison
        max_r2_XGB = max(result['Inner R2'] for result in inner_results_XGB)
        candidates_XGB = [result for result in inner_results_XGB if abs(result['Inner R2'] - max_r2_XGB) < epsilon]
        best_inner_XGB = min(candidates_XGB, key=lambda x: x['Inner RMSE'])
        best_n_features_XGB = best_inner_XGB['Num Features']
        best_params_XGB = best_inner_XGB['Best Params']
        best_top_indices_XGB = best_inner_XGB['Top Indices']
    else:
        print(f"No valid inner results for Outer Fold {outer_fold}. Skipping this fold.")
        continue
    
    # Train the final model for this outer fold on full outer training data using selected features
    X_train_best_XGB = X_train_outer_XGB.iloc[:, best_top_indices_XGB]
    final_model_XGB = RandomForestRegressor(**best_params_XGB)
    final_model_XGB.fit(X_train_best_XGB, y_train_outer_XGB)
    
    # Evaluate on the outer test set using the same selected features
    X_test_best_XGB = X_test_outer_XGB.iloc[:, best_top_indices_XGB]
    y_pred_outer_XGB = final_model_XGB.predict(X_test_best_XGB)
    outer_r2_XGB = r2_score(y_test_outer_XGB, y_pred_outer_XGB)
    outer_rmse_XGB = np.sqrt(mean_squared_error(y_test_outer_XGB, y_pred_outer_XGB))
    outer_mae_XGB = mean_absolute_error(y_test_outer_XGB, y_pred_outer_XGB)
    
    print(f"Outer Fold {outer_fold} Final Model: n_features={best_n_features_XGB}, Outer R2={outer_r2_XGB:.3f}, Outer RMSE={outer_rmse_XGB:.3f}")
    
    outer_results_XGB.append({
        'Fold': outer_fold,
        'Num Features': best_n_features_XGB,
        'Best Params': best_params_XGB,
        'Outer R2': outer_r2_XGB,
        'Outer RMSE': outer_rmse_XGB,
        'Top Indices': best_top_indices_XGB,
        'Outer Train Indices': train_index,
        'Outer Test Indices': test_index,
        'Outer MAE': outer_mae_XGB,
    })

# ============================================================
# 4) Summarize Outer CV Results
# ============================================================
outer_results_df_XGB = pd.DataFrame(outer_results_XGB)
print("\n=== Outer CV Results ===")
print(outer_results_df_XGB)
avg_outer_r2_XGB = outer_results_df_XGB['Outer R2'].mean()
avg_outer_rmse_XGB = outer_results_df_XGB['Outer RMSE'].mean()
avg_outer_mae_XGB = outer_results_df_XGB['Outer MAE'].mean()
print(f"\nAverage Outer R2: {avg_outer_r2_XGB:.3f}")
print(f"Average Outer RMSE: {avg_outer_rmse_XGB:.3f}")
print(f"Average Outer MAE: {avg_outer_mae_XGB:.3f}")

# ============================================================
# 5) Create Hold-out Set and Retrain Final Model on Full Training Data
# ============================================================
# Split the full dataset (X_XGB, y_XGB) into training and hold-out sets (e.g., 80% train, 20% hold-out)
X_train_full, X_holdout, y_train_full, y_holdout = train_test_split(X_XGB, y_XGB, test_size=0.2, random_state=42)

# Use the optimal feature indices from the best overall outer fold configuration
best_overall_XGB = outer_results_df_XGB.loc[outer_results_df_XGB['Outer R2'].idxmax()]
best_n_features_overall_XGB = best_overall_XGB['Num Features']
best_params_overall_XGB = best_overall_XGB['Best Params']
best_top_indices_overall_XGB = best_overall_XGB['Top Indices']

print("\nBest Overall Outer Fold Configuration:")
print(best_overall_XGB)

# Create optimal feature subsets for both training and hold-out sets
X_train_full_optimal = X_train_full.iloc[:, best_top_indices_overall_XGB].copy()
X_holdout_optimal = X_holdout.iloc[:, best_top_indices_overall_XGB].copy()

best_cols = X_train_full.columns[best_top_indices_overall_XGB]

X_train_full_optimal = X_train_full[best_cols].copy()
X_holdout_optimal   = X_holdout[best_cols].copy()
# Retrain the final model on the full training data
final_model_full_XGB = RandomForestRegressor(**best_params_overall_XGB)
final_model_full_XGB.fit(X_train_full_optimal, y_train_full)

# Make predictions on both training and hold-out sets (predictions are in transformed scale)
y_train_pred_final = final_model_full_XGB.predict(X_train_full_optimal)
y_holdout_pred_final = final_model_full_XGB.predict(X_holdout_optimal)

# --------------------------------------------------
# Inverse transform predictions and true target values to original scale
# --------------------------------------------------
# It is assumed that the PowerTransformer 'pt' is still available.
# Convert arrays back to DataFrames with the original column name used during fitting.
col_name = 'Cc'  # Change this to the column name used when fitting pt

y_train_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_train_pred_final.reshape(-1, 1), columns=[col_name])
)
# Inverse-transform predictions (hold‑out and train) and true targets
y_holdout_pred_final_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout_pred_final.reshape(-1, 1), columns=[col_name])
)

# Convert y_train_full (pandas Series/DataFrame) to numpy then reshape
y_train_full_orig = pt.inverse_transform(
    pd.DataFrame(y_train_full.reshape(-1, 1), columns=[col_name])
)

y_holdout_orig = pt.inverse_transform(
    pd.DataFrame(y_holdout.reshape(-1, 1), columns=[col_name])
)


# Evaluate performance on original scale
train_r2_final_orig = r2_score(y_train_full_orig, y_train_pred_final_orig)
train_rmse_final_orig = np.sqrt(mean_squared_error(y_train_full_orig, y_train_pred_final_orig))
train_mae_final_orig = mean_absolute_error(y_train_full_orig, y_train_pred_final_orig)
holdout_r2_final_orig = r2_score(y_holdout_orig, y_holdout_pred_final_orig)
holdout_rmse_final_orig = np.sqrt(mean_squared_error(y_holdout_orig, y_holdout_pred_final_orig))
holdout_mae_final_orig = mean_absolute_error(y_holdout_orig, y_holdout_pred_final_orig)

print(f"\nFinal Model Training Performance (Original Scale): Train R2 = {train_r2_final_orig:.3f}, Train RMSE = {train_rmse_final_orig:.3f}, Train MAE = {train_mae_final_orig:.3f}")
print(f"Final Model Hold-out Performance (Original Scale): Holdout R2 = {holdout_r2_final_orig:.3f}, Holdout RMSE = {holdout_rmse_final_orig:.3f}, Holdout MAE = {holdout_mae_final_orig:.3f}")

# ============================================================
# 6) Combined Regression Plot (Final Model on Full Training Data)
# ============================================================
plt.figure(figsize=(8, 6))
plt.scatter(y_train_full_orig, y_train_pred_final_orig, alpha=0.7, color="blue",
            label=f"Training Data\nRMSE: {train_rmse_final_orig:.2f}, R²: {train_r2_final_orig:.2f}, MAE: {train_mae_final_orig:.2f}")
plt.scatter(y_holdout_orig, y_holdout_pred_final_orig, alpha=0.7, color="green",
            label=f"Hold-out Data\nRMSE: {holdout_rmse_final_orig:.2f}, R²: {holdout_r2_final_orig:.2f}, MAE: {holdout_mae_final_orig:.2f}")
lims = [min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()),
        max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max())]
plt.plot(lims, lims, 'r--', label='Perfect Prediction')
plt.xlim(lims)
plt.ylim(lims)
min_val = int(min(y_train_full_orig.min(), y_holdout_orig.min(), y_train_pred_final_orig.min(), y_holdout_pred_final_orig.min()))
max_val = int(max(y_train_full_orig.max(), y_holdout_orig.max(), y_train_pred_final_orig.max(), y_holdout_pred_final_orig.max()))
plt.xticks(np.arange(min_val, max_val + 1, 1))
plt.yticks(np.arange(min_val, max_val + 1, 1))
plt.title("Actual vs Predicted Values (Final Model on Training & Hold-out Data)")
plt.xlabel("Actual Values (Original Scale)")
plt.ylabel("Predicted Values (Original Scale)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig("combined_parity_plot_nested_XGB_holdout.png")
plt.show()

# ============================================================
# 7) SHAP Analysis on the Final Model (Trained on Full Training Data)
# ============================================================
explainer_final = shap.TreeExplainer(final_model_full_XGB)
shap_values_final_XGB = explainer_final.shap_values(X_train_full_optimal)

# (a) SHAP Beeswarm Plot
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, show=False)
plt.title("SHAP Beeswarm - Final Model (Training Data)")
plt.savefig("shap_summary_beeswarm_nested_XGB_holdout.png")
plt.show()

# (b) SHAP Bar Plot (global mean(|SHAP|))
shap.summary_plot(shap_values_final_XGB, features=X_train_full_optimal,
                  feature_names=X_train_full_optimal.columns, plot_type="bar", show=False)
plt.title("SHAP Bar Plot (Final Model)")
plt.savefig("shap_summary_bar_nested_XGB_holdout.png")
plt.show()

# (c) Optional: Waterfall Plot for a single sample (index 0) from training data
sample_index = 0
base_value = explainer_final.expected_value  # Baseline (average model output)
sample_shap = shap_values_final_XGB[sample_index, :]
try:
    shap.plots.waterfall(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index}")
    plt.savefig("shap_waterfall_sample0_nested_XGB_holdout.png")
    plt.show()
except AttributeError:
    shap.waterfall_plot(
        shap.Explanation(
            base_values=base_value,
            values=sample_shap,
            data=X_train_full_optimal.iloc[sample_index, :],
            feature_names=X_train_full_optimal.columns
        )
    )
    plt.title(f"SHAP Waterfall for Sample Index={sample_index} (Legacy)")
    plt.savefig("shap_waterfall_sample0_legacy_nested_XGB_holdout.png")
    plt.show()

print("\nDone! Created:")
print("  - combined_parity_plot_nested_XGB_holdout.png")
print("  - shap_summary_beeswarm_nested_XGB_holdout.png")

print("  - shap_summary_bar_nested_XGB_holdout.png")
print("  - shap_waterfall_sample0_nested_XGB_holdout.png")